# Homer and Li's two-line table: the joint occurrence severity and the annual conditional means

The three events of Homer and Li 2017, Table 11, split into personal (P) and commercial (C) lines, through the multi-unit route of `eltq` (spec section 4A; the multi-unit article, sections on the joint distribution and on the annual bivariate aggregate). Two of the six rows are two-point rows.

1. The per-view totals and the rate-weighted joint occurrence severity of the two lines (`multiunit_views`, `occurrence_pair_joint`).
2. The annual conditional mean of each line given the portfolio total, $\kappa_u(s) = \mathsf P(A_u \mid A = s)$, by the Mecke identity from the first-moment vectors (`occurrence_unit_first_moment`, `annual_conditional_mean`), a one-dimensional convolution.
3. The same $\kappa_P$ by the two-dimensional route: the sheared joint compounded by the aggregate package's `BivariateAggregate` and read off with `exeqa_df`. This section needs the `aggregate` package.


In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

import eltq

H = 10.0
# Homer and Li 2017, Table 11: (mean, sd_indep, sd_corr, exposed value) per line.
homer = {
    1: (0.1, (300.0, 400.0, 300.0, 3000.0), (200.0, 300.0, 200.0, 1000.0)),
    2: (0.1, (100.0, 371.0, 267.0, 1000.0), (200.0, 150.0, 533.0, 4000.0)),
    3: (0.5, (100.0, 224.0, 200.0, 2000.0), (100.0, 200.0, 200.0, 2000.0)),
}
rows = []
for event, (rate, personal, commercial) in homer.items():
    rows.append((f"e{event}", "P", rate, *personal))
    rows.append((f"e{event}", "C", rate, *commercial))
elt = pd.DataFrame(rows, columns=["event_id", "unit_id", "rate", "mean_loss", "sd_indep", "sd_corr", "max_loss"])
columns = [elt[c].tolist() if c in ("event_id", "unit_id") else elt[c].to_numpy() for c in elt.columns]
elt

## The portfolio total and the joint occurrence severity

Under the default coupling (Burke's loading on the common factor) and the stratified factor rule. The two inadmissible rows are reported under `variance_bernoulli`.

In [ ]:
views, meta = eltq.multiunit_views(*columns, bucket_size=H, views=("ground_up",))
severity = views["ground_up"]            # rate-weighted occurrence severity of the total
x = H * np.arange(meta.grid_length)
print(f"total rate {meta.total_rate}, mean {meta.view_mean['ground_up']:.4f} (analytic {meta.analytic_aal}),"
      f" coupling {meta.coupling}, rule {meta.z_rule} with {meta.z_strata} strata")
print(meta.diagnostics)

pmf, x_p, x_c, pair = eltq.occurrence_pair_joint(*columns, units=("P", "C"), bucket_size=H)
print(f"pair mass {pair.mass} over {pair.events} events; occurrence correlation {pair.grid_correlation:.3f}")

fig, ax = plt.subplots(figsize=(6, 4))
with np.errstate(divide="ignore"):
    ax.contour(x_c, x_p, np.log10(pmf / pair.mass), levels=np.arange(-5, -2.4, 0.25))
ax.set(xlabel="commercial loss", ylabel="personal loss", title="occurrence joint, log10 cell probability")
plt.show()

## The annual conditional means by the Mecke identity

$m_u(y) = \sum_j \lambda_j \mathsf P(X_{uj}; X_j = y)$ is the rate-weighted first moment of unit $u$ at each occurrence total; $\sum_u m_u(y) = y\, s(y)$. With the annual density $f_A$ of the total (here a compound Poisson by FFT on the severity above; the aggregate package gives the same), $\kappa_u(s) = (m_u * f_A)(s) / f_A(s)$.

In [ ]:
moments, x_m, fm = eltq.occurrence_unit_first_moment(*columns, bucket_size=H)
print(f"identity error {fm.identity_error:.1e}; sum of first moments {fm.total_mean:.4f}")

size = 4096
f_a = np.fft.irfft(np.exp(np.fft.rfft(severity, size) - severity.sum()), size)   # annual density, Poisson counts
s = H * np.arange(size)
kappa = eltq.annual_conditional_mean(moments, f_a)

# At an occurrence: kappa_u(y) = m_u(y) / s(y).
with np.errstate(invalid="ignore", divide="ignore"):
    kappa_occ = {u: m / severity for u, m in moments.items()}

fig, axs = plt.subplots(1, 2, figsize=(10, 4.5))
live = severity > 1e-12 * severity.sum()
axs[0].plot(x[live], kappa_occ["P"][live], label=r"$\kappa_P$, personal")
axs[0].plot(x[live], kappa_occ["C"][live], label=r"$\kappa_C$, commercial")
axs[0].plot(x[live], x[live], "k--", lw=0.6, label="their sum")
axs[0].set(xlabel="total loss of the occurrence", ylabel="conditional mean", title="at an occurrence")
annual = f_a > 1e-9
axs[1].plot(s[annual], kappa["P"][annual], label=r"$\kappa_P$")
axs[1].plot(s[annual], kappa["C"][annual], label=r"$\kappa_C$")
axs[1].plot(s[annual], s[annual], "k--", lw=0.6)
axs[1].set(xlabel="annual portfolio loss", title="in a year")
axs[0].legend(fontsize="small")
plt.show()

## The two-dimensional route, for comparison

The sheared joint $(X_P, X_P + X_C)$ compounded in two dimensions by the aggregate package, and $\kappa_P$ read off with `exeqa_df`. The one-dimensional route above agrees wherever the two-dimensional grid does not wrap. Skip this cell without the `aggregate` package.

In [ ]:
from aggregate.bivariate import BivariateAggregate

n_p, n_c = pmf.shape
sheared = np.zeros((n_p, n_p + n_c - 1))
for a in range(n_p):
    sheared[a, a:a + n_c] = pmf[a]
bv = BivariateAggregate("homer", mode="discrete", dbv_xs=x_p, dbv_ys=H * np.arange(n_p + n_c - 1),
                        dbv_S=sheared / pair.mass, freq_name="poisson", exp_en=pair.mass)
bv.update(log2=(10, 11), bs=(H, H))
kappa_df = bv.exeqa_df(axis=1)
kd = kappa_df[(kappa_df.p > 1e-9) & (kappa_df.index < 4000)]
index = (kd.index.to_numpy() / H).round().astype(int)
print("largest difference of the two routes below 4,000:", np.abs(kappa["P"][index] - kd.exeqa_X.to_numpy()).max())